In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
import json
import re
import os
from unicodedata import normalize
#from .unipath_gdg import UNIPATH_GDG

In [2]:
def preprocess_chunks(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    documents = []
    for chunk in data:
        text = chunk['text']
        # Clean excessive whitespace/newlines
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) > 50:  # Filter meaningful chunks
            documents.append({
                'text': text,
                'metadata': chunk.get('metadata', {}),
                'chunk_id': chunk.get('chunk_id')
            })
    return documents

In [17]:
# Example usage
file_path = 'data/rag_dataset_fixed.json' if os.path.exists('data/rag_dataset_fixed.json') else 'rag_dataset_fixed.json'
documents = preprocess_chunks(file_path)
df = pd.DataFrame(documents)


FileNotFoundError: [Errno 2] No such file or directory: 'rag_dataset_fixed.json'

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Initialize Arabic/English bilingual model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Generate embeddings
embeddings = model.encode([doc['text'] for doc in documents])

# Create FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))

In [ ]:
# Example usage


In [ ]:
def retrieve(query, k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k)
    
    results = []
    for idx in indices[0]:
        results.append({
            'text': documents[idx]['text'],
            'metadata': documents[idx]['metadata'],
            'score': 1 - distances[0][list(indices[0]).index(idx)]/2  # cosine similarity approx
        })
    return results

In [ ]:
def rag_query(query, llm_client):
    # Retrieve relevant chunks
    context_chunks = retrieve(query, k=3)
    context = "\n\n".join([f"Document {i+1}: {c['text']}" for i, c in enumerate(context_chunks)])
    
    # Construct prompt with academic regulations context
    prompt = f"""You are an academic advisor for Helwan University Faculty of Science.
Answer based ONLY on the provided context. If information is not in context, say "Not specified in regulations."

Context:
{context}

Question: {query}

Answer in formal academic Arabic/English as appropriate:"""
    
    response = llm_client.generate(prompt)
    return {
        'answer': response,
        'sources': [c['metadata'].get('source', 'N/A') for c in context_chunks]
    }